In [11]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("records").getOrCreate()


25/03/07 09:36:23 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [17]:
 spark.sql("DROP TABLE IF EXISTS demo.nyc.taxis_1K_product_p2_new")


DataFrame[]

In [18]:
import os
iceberg_table_dir = "../warehouse/nyc/taxis_1K_product_p2_new"
metadata_dir = f"{iceberg_table_dir}/metadata"
data_dir = f"{iceberg_table_dir}/data"
input_data_dir = f"../input_data"
analysis_info = []
records_before_op = 0

def append_to_file(file_path, msg):
    open_mode = "a"
    if not os.path.exists(file_path):
        open_mode = "w"

    # Open the CSV file in write mode
    with open(file_path, open_mode) as file:
        writer = csv.writer(file)
        
        if open_mode=="w":
            #writing header of the columns
            writer.writerows([list(msg.keys())])    

        row_values = [list(msg.values())]
        # Write the data to the CSV file
        writer.writerows(row_values)

def get_size():
    # List the metadata files
    manifest_pattern = re.compile(r".*-m\d+\.avro$")
    metadata_files = os.listdir(metadata_dir)
    
    # Initialize variables to store the sizes of different types of metadata files
    snap_avro_size = 0
    metadata_json_size = 0
    m_avro_size = 0

    data_dir_size = 0
    # get data dir size
    data_dir_files = os.listdir(data_dir)
    # print(data_dir_files)
    for filename in data_dir_files:
        file_path = os.path.join(data_dir, filename)
        data_dir_size += os.path.getsize(file_path) / 1024  # Convert size to KB
    
    # Iterate through the metadata files and calculate their sizes
    for file in metadata_files:
        file_path = os.path.join(metadata_dir, file)
        file_size_kb = os.path.getsize(file_path) / 1024  # Convert size to KB
        
        if file.startswith("snap-") and file.endswith(".avro"):
            snap_avro_size += file_size_kb
        elif file.endswith(".metadata.json"):
            metadata_json_size += file_size_kb
        elif manifest_pattern.match(file):
            m_avro_size += file_size_kb
    
    # Print the time taken and the sizes of the metadata files
    # print(f"Time taken to read Parquet files: {time_taken:.2f} seconds")
    # print(f"Size of snap-*.avro files: {snap_avro_size:.2f} KB")
    # print(f"Size of *.metadata.json files: {metadata_json_size:.2f} KB")
    # print(f"Size of *m{0-9}{1,}.avro files: {m_avro_size:.2f} KB")

    return {"data_dir_size": data_dir_size,"metadata_size": metadata_json_size,"snapshot_size": snap_avro_size,"manifest_size": m_avro_size}


In [19]:
import random
import string
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DateType
)
from datetime import datetime, timedelta

# Initialize Spark session
spark = SparkSession.builder.appName("Generate Taxis Data").getOrCreate()

def generate_random_month_date():
    """Generate a random date with the first day of a month."""
    start_date = datetime(2020, 1, 1)
    end_date = datetime.today()
    delta = end_date - start_date
    random_days = random.randint(0, delta.days)
    random_date = start_date + timedelta(days=random_days)
    
    return random_date.replace(day=1)  # Ensuring DATE format

# Define schema ensuring extra_col_3 is DateType
schema = StructType([
    StructField(f"extra_col_{i}", StringType(), True) if i % 4 == 0 else  
    StructField(f"extra_col_{i}", IntegerType(), True) if i % 4 == 1 else  
    StructField(f"extra_col_{i}", StringType(), True) if i % 4 == 2 else  
    StructField(f"extra_col_{i}", DateType(), True)  # Ensure DATE type for extra_col_3
    for i in range(50)
])

def generate_records(n):
    """Generate records based on schema."""
    data = []
    for _ in range(n):
        row = [
            ''.join(random.choice(string.ascii_uppercase) for _ in range(10)) if i % 4 == 0 else
            random.randint(1, 10000) if i % 4 == 1 else
            random.choice(["CAR", "TW", "CV"]) if i % 4 == 2 else
            generate_random_month_date()  # Ensure DATE type
            for i in range(50)
        ]
        data.append(row)
    
    return data

# Generate Data
records = generate_records(1000)  # Adjust number of records

# Create DataFrame
df = spark.createDataFrame(records, schema)

# Ensure extra_col_3 is stored as DateType
df = df.withColumn("extra_col_3", F.to_date(F.col("extra_col_3")))

# Show sample records
df.show(5)

# Save to Iceberg Table
# df.writeTo("demo.nyc.taxis_1000_50_product").create()


df.writeTo("demo.nyc.taxis_1K_product_p2_new") \
    .partitionedBy("extra_col_6") \
    .create()



+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+-----------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+------------+
|extra_col_0|extra_col_1|extra_col_2|extra_col_3|extra_col_4|extra_col_5|extra_col_6|extra_col_7|extra_col_8|extra_col_9|extra_col_10|extra_col_11|extra_col_12|extra_col_13|extra_col_14|extra_col_15|extra_col_16|extra_col_17|extra_col_18|extra_col_19|extra_col_20|extra_col_21|extra_col_22|extra_col_23|extra_col_24|extra_col_25|extra_col_26|extra_col_27|ext

In [20]:
import time

# Measure time for select operation
start_time = time.time()
df.select("extra_col_0" ,"extra_col_1", "extra_col_6").show(5)
select_time = time.time() - start_time
print(f"Select operation time: {select_time} seconds")


+-----------+-----------+-----------+
|extra_col_0|extra_col_1|extra_col_6|
+-----------+-----------+-----------+
| RGDVHSUZWB|       6870|         CV|
| HLGASBZOHY|       7650|         TW|
| LLUFSTKNVR|       1929|        CAR|
| MICYGTNKTK|       9426|         TW|
| DVPYOEPBBY|       4396|        CAR|
+-----------+-----------+-----------+
only showing top 5 rows

Select operation time: 0.1692206859588623 seconds


In [21]:
import time

# Measure time for select operation
start_time = time.time()
df22=df.select("extra_col_0", "extra_col_2")
df22.show(5)
select_time = time.time() - start_time
print(f"Select operation time: {select_time} seconds")

+-----------+-----------+
|extra_col_0|extra_col_2|
+-----------+-----------+
| RGDVHSUZWB|        CAR|
| HLGASBZOHY|         CV|
| LLUFSTKNVR|         TW|
| MICYGTNKTK|         CV|
| DVPYOEPBBY|         TW|
+-----------+-----------+
only showing top 5 rows

Select operation time: 0.19242501258850098 seconds


In [22]:
# Measure time for join operation
# Measure time for join operation
start_time = time.time()

# Perform the join and alias the DataFrames
joined_df_new = df.alias("df").join(df22.alias("df22"), "extra_col_0") \
    .select("df.extra_col_1", "df22.extra_col_0")

# Show the first 5 rows of the result
joined_df_new.show(5)

# Measure time for the join operation
join_time = time.time() - start_time
print(f"Join operation time: {join_time} seconds")


+-----------+-----------+
|extra_col_1|extra_col_0|
+-----------+-----------+
|       6870| RGDVHSUZWB|
|       9023| CQTCEEQDXK|
|       3037| WIVOFLDICK|
|       8962| WEDHWPSLBT|
|       4271| HNEBEVBANW|
+-----------+-----------+
only showing top 5 rows

Join operation time: 0.5781693458557129 seconds


In [23]:
# Measure time for aggregate operation
start_time = time.time()
aggregated_df_new = df.groupBy("extra_col_6").agg({"extra_col_1": "avg"})
aggregated_df_new.show()
aggregate_time = time.time() - start_time
print(f"Aggregate operation time: {aggregate_time} seconds")


+-----------+-----------------+
|extra_col_6| avg(extra_col_1)|
+-----------+-----------------+
|        CAR|         5115.625|
|         CV|5263.821752265861|
|         TW|4969.953079178886|
+-----------+-----------------+

Aggregate operation time: 0.48717665672302246 seconds
